In [3]:
# Dask puts out more advisory logging that we care for.
# It takes some doing to quiet all of it, but this recipe works.
import dask
import logging
import dask_jobqueue
from dask.dataframe.utils import make_meta
from dask.distributed import Client

dask.config.set({"logging.distributed": "critical"})

# This also has to be done, for the above to be effective
logger = logging.getLogger("distributed")
logger.setLevel(logging.CRITICAL)

import os
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import warnings

# Finally, suppress the specific warning about Dask dashboard port usage
warnings.filterwarnings("ignore", message="Port 8787 is already in use.")

from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt
import time

from hats import read_hats

import lsdb

from catalog_filtering import bandFilterLenient, contains_PM
import hpms_pipeline as hpms

print("Imported libraries.")

Imported libraries.


In [5]:
BENCHMARK_CATALOG_DIR = Path("../../../../catalogs/benchmark_catalogs")
RESULTS_DIR = BENCHMARK_CATALOG_DIR / 'two_deg_cs_2_results'
two_deg_results = lsdb.read_hats(RESULTS_DIR)
with Client(): 
    computed_results = two_deg_results.compute()
    
filtered_size = len(computed_results)
print(f"Length of filtered subset: {filtered_size}")
computed_results

Length of filtered subset: 172056


,COADD_OBJECT_ID_1,CLASS_STAR_G_2,CLASS_STAR_R_2,CLASS_STAR_I_2,CLASS_STAR_Z_2,CLASS_STAR_Y_2,FLAGS_G_2,FLAGS_R_2,FLAGS_I_2,FLAGS_Z_2,FLAGS_Y_2,RA_2,DEC_2,COADD_OBJECT_ID_2,SPREAD_MODEL_G_2,SPREAD_MODEL_R_2,SPREAD_MODEL_I_2,SPREAD_MODEL_Z_2,SPREAD_MODEL_Y_2,WAVG_MAG_PSF_G_2,WAVG_MAG_PSF_R_2,WAVG_MAG_PSF_I_2,WAVG_MAG_PSF_Z_2,WAVG_MAG_PSF_Y_2,WAVG_MAGERR_PSF_G_2,WAVG_MAGERR_PSF_R_2,WAVG_MAGERR_PSF_I_2,WAVG_MAGERR_PSF_Z_2,WAVG_MAGERR_PSF_Y_2,NEPOCHS_G_2,NEPOCHS_R_2,NEPOCHS_I_2,NEPOCHS_Z_2,NEPOCHS_Y_2,kth_min_deviation,max_obj_distance,max_mag_diff
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1153199886535753567,1031956367,0.384846,0.69309,0.641284,0.245487,0.44507,0,0,0,0,0,0.065315,-38.894643,1031956468,-0.004457,0.008173,0.002803,0.004519,-0.012644,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,0.14189,30.486081,<NA>
1153198550409663336,1031956443,0.695838,0.749122,0.669979,0.586488,0.485705,0,0,0,0,0,359.808644,-38.907508,1031958638,0.008227,-0.000507,0.001995,-0.004563,0.0594,-99.0,23.584322,-99.0,-99.0,-99.0,-99.0,0.280294,-99.0,-99.0,-99.0,0,1,0,0,0,0.156544,34.915324,0.478099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499136843076061954,1051599127,0.009067,0.013581,0.016127,0.008689,0.001903,0,0,0,0,0,3.659195,-38.178604,1051122243,0.016997,0.015365,0.013966,0.013957,0.003196,23.992067,23.423948,22.813665,22.501387,-99.0,0.09049,0.054621,0.049008,0.07276,-99.0,4,7,8,5,0,0.086319,20.682652,17.52524
2499142425183615388,1051599153,0.387475,0.459344,0.538657,0.436509,0.474185,0,0,0,0,0,3.50259,-38.178026,1051122795,0.007815,-0.000743,0.00306,0.006273,-0.093401,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,0.117117,19.225094,13.958419


In [6]:
filtered_results = computed_results.query('kth_min_deviation < 0.06 and max_mag_diff < 10 and max_obj_distance < 24')
filtered_results

,COADD_OBJECT_ID_1,CLASS_STAR_G_2,CLASS_STAR_R_2,CLASS_STAR_I_2,CLASS_STAR_Z_2,CLASS_STAR_Y_2,FLAGS_G_2,FLAGS_R_2,FLAGS_I_2,FLAGS_Z_2,FLAGS_Y_2,RA_2,DEC_2,COADD_OBJECT_ID_2,SPREAD_MODEL_G_2,SPREAD_MODEL_R_2,SPREAD_MODEL_I_2,SPREAD_MODEL_Z_2,SPREAD_MODEL_Y_2,WAVG_MAG_PSF_G_2,WAVG_MAG_PSF_R_2,WAVG_MAG_PSF_I_2,WAVG_MAG_PSF_Z_2,WAVG_MAG_PSF_Y_2,WAVG_MAGERR_PSF_G_2,WAVG_MAGERR_PSF_R_2,WAVG_MAGERR_PSF_I_2,WAVG_MAGERR_PSF_Z_2,WAVG_MAGERR_PSF_Y_2,NEPOCHS_G_2,NEPOCHS_R_2,NEPOCHS_I_2,NEPOCHS_Z_2,NEPOCHS_Y_2,kth_min_deviation,max_obj_distance,max_mag_diff
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1153199464009600005,1031960709,0.704812,0.119973,0.142955,0.911769,0.592972,0,0,0,0,0,0.008249,-38.931671,1031960720,0.006649,0.016757,0.013554,-0.013997,-0.014617,-99.0,-99.0,23.428898,-99.0,-99.0,-99.0,-99.0,0.135526,-99.0,-99.0,0,0,3,0,0,0.017987,19.974376,0.492011
1153199464089760270,1031960720,0.460136,0.326873,0.378898,0.087728,0.405223,0,0,0,0,0,0.009109,-38.931538,1031960709,0.012267,0.01237,0.005402,0.000607,-0.022767,-99.0,-99.0,-99.0,22.942133,-99.0,-99.0,-99.0,-99.0,0.211099,-99.0,0,0,0,1,0,0.017987,19.974376,0.492011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2499142491409908258,1051596981,0.112115,0.014347,0.08699,0.016111,0.000331,0,0,0,0,0,3.519712,-38.153892,1051597020,0.012654,0.014592,0.01129,0.015557,0.015632,-99.0,23.388672,22.742029,22.369293,-99.0,-99.0,0.051605,0.046546,0.052826,-99.0,0,7,8,9,0,0.03489,21.691004,4.836164
2499142583558307712,1051597020,0.000322,0.000326,0.012117,0.00036,0.000245,3,3,3,3,3,3.521685,-38.155185,1051596981,-0.005563,-0.00595,0.018683,0.015823,0.011156,-99.0,23.359688,22.790199,22.181501,-99.0,-99.0,0.097346,0.053563,0.081505,-99.0,0,2,5,4,0,0.034891,21.691004,4.836164
